# 81 — RFP Response Assistant

## Goal
Parse an **RFP (Request for Proposal)**, retrieve **prior answers**
from a knowledge base, **draft tailored responses**, and **flag unknowns**
that need human review.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **LangGraph** | Multi-step stateful workflow |
| **RAG** | Retrieve relevant past answers |
| **Routing** | Decide answer vs flag-for-human |
| **Structured state** | TypedDict graph state |

## Stack
- `langgraph` — stateful agent graph
- `langchain-ollama` — LLM via Ollama
- `chromadb` — vector store for past answers
- Ollama `qwen3.5:9b`

In [ ]:
import os, warnings, json
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_NO_TF"] = "1"
warnings.filterwarnings("ignore")

from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
import chromadb

llm = ChatOllama(model="qwen3.5:9b", temperature=0)
print("All imports OK")

## Step 1 — Past Answers Knowledge Base

Simulate a database of **previously answered RFP questions**
that we can retrieve from.

In [ ]:
# Past RFP answers — our knowledge base
PAST_ANSWERS = [
    {
        "question": "What is your company's data security policy?",
        "answer": "We implement AES-256 encryption at rest and TLS 1.3 in transit. All data is stored in SOC 2 Type II certified data centers. We conduct annual penetration testing and quarterly vulnerability assessments.",
        "category": "security"
    },
    {
        "question": "Describe your disaster recovery plan.",
        "answer": "Our DR plan includes RPO of 1 hour and RTO of 4 hours. We maintain hot standby in a geographically separate region. Automated failover is tested quarterly. Full DR drills are conducted annually.",
        "category": "security"
    },
    {
        "question": "What is your uptime SLA?",
        "answer": "We guarantee 99.95% uptime SLA for enterprise tier. This excludes scheduled maintenance windows (max 4 hours/month, announced 72 hours in advance). SLA credits are issued automatically.",
        "category": "operations"
    },
    {
        "question": "Describe your onboarding and implementation process.",
        "answer": "Implementation follows a 4-phase approach: Discovery (2 weeks), Configuration (3 weeks), UAT (2 weeks), Go-Live (1 week). A dedicated implementation manager is assigned. We provide comprehensive training for up to 20 users.",
        "category": "operations"
    },
    {
        "question": "What compliance certifications do you hold?",
        "answer": "We hold SOC 2 Type II, ISO 27001, GDPR compliance, and HIPAA BAA. Certifications are renewed annually. Audit reports are available under NDA upon request.",
        "category": "compliance"
    },
    {
        "question": "Describe your support tiers.",
        "answer": "We offer 3 support tiers: Standard (email, 24h response), Premium (email + chat, 4h response), Enterprise (24/7 phone + dedicated CSM, 1h response for P1). All tiers include access to our knowledge base.",
        "category": "operations"
    },
    {
        "question": "What integrations do you support?",
        "answer": "We offer REST APIs, webhooks, and pre-built connectors for Salesforce, SAP, Oracle, and Microsoft 365. Custom integrations are available via our SDK. API documentation is public.",
        "category": "technical"
    },
]

# Build ChromaDB vector store
chroma_client = chromadb.Client()
collection = chroma_client.create_collection("past_rfp_answers")

collection.add(
    documents=[a["question"] + " " + a["answer"] for a in PAST_ANSWERS],
    metadatas=[{"category": a["category"]} for a in PAST_ANSWERS],
    ids=[f"ans_{i}" for i in range(len(PAST_ANSWERS))]
)

print(f"Loaded {collection.count()} past answers into vector store")

## Step 2 — Sample RFP Document

In [ ]:
SAMPLE_RFP = """
REQUEST FOR PROPOSAL: Enterprise SaaS Platform
Issued by: Acme Financial Services
Date: 2024-10-15

Section 3: Technical & Security Requirements

Q1: Please describe your data encryption standards and security certifications.

Q2: What is your guaranteed uptime and how do you handle service disruptions?

Q3: Describe your API capabilities and third-party integration support.

Q4: What is your approach to AI/ML model governance and bias detection?

Q5: Describe your customer support structure and escalation procedures.
"""

print(SAMPLE_RFP)

## Step 3 — LangGraph Workflow

```
parse_rfp → retrieve_answers → draft_responses → review_flags → END
```

In [ ]:
# Graph state definition
class RFPState(TypedDict):
    rfp_text: str
    questions: list          # parsed questions
    retrieved_context: list  # RAG results per question
    draft_responses: list    # drafted answers
    flagged_items: list      # questions needing human review
    final_output: str        # formatted output

print("State schema defined")

In [ ]:
# Node 1: Parse RFP into individual questions
def parse_rfp(state: RFPState) -> RFPState:
    print("📄 Parsing RFP...")
    
    msg = llm.invoke([
        SystemMessage(content="Extract all questions from this RFP document. Return ONLY a JSON array of strings, one per question. No extra text."),
        HumanMessage(content=state["rfp_text"])
    ])
    
    raw = msg.content
    # Strip thinking tags
    if "<think>" in raw:
        raw = raw.split("</think>")[-1].strip()
    # Extract JSON array
    if "```" in raw:
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()
    
    try:
        questions = json.loads(raw)
    except json.JSONDecodeError:
        # Fallback: split by Q pattern
        import re
        questions = re.findall(r'Q\d+:\s*(.+)', state["rfp_text"])
    
    print(f"  Found {len(questions)} questions")
    for i, q in enumerate(questions):
        print(f"  Q{i+1}: {q[:80]}")
    
    return {**state, "questions": questions}

print("Node: parse_rfp defined")

In [ ]:
# Node 2: Retrieve relevant past answers
def retrieve_answers(state: RFPState) -> RFPState:
    print("\n🔍 Retrieving past answers...")
    
    retrieved = []
    for q in state["questions"]:
        results = collection.query(query_texts=[q], n_results=2)
        docs = results["documents"][0] if results["documents"] else []
        distances = results["distances"][0] if results["distances"] else []
        
        retrieved.append({
            "question": q,
            "similar_answers": docs,
            "distances": distances,
            "has_good_match": len(distances) > 0 and distances[0] < 1.2
        })
        match_str = "✅ Good match" if retrieved[-1]["has_good_match"] else "⚠️ Weak match"
        print(f"  {q[:60]}... → {match_str} (dist={distances[0]:.2f})")
    
    return {**state, "retrieved_context": retrieved}

print("Node: retrieve_answers defined")

In [ ]:
# Node 3: Draft responses using LLM + retrieved context
def draft_responses(state: RFPState) -> RFPState:
    print("\n✍️ Drafting responses...")
    
    drafts = []
    flagged = []
    
    for ctx in state["retrieved_context"]:
        q = ctx["question"]
        
        if ctx["has_good_match"]:
            # Draft based on past answers
            past = "\n".join(ctx["similar_answers"])
            msg = llm.invoke([
                SystemMessage(content="""You are an RFP response writer. Using the past answers as reference,
draft a professional response to the RFP question. Tailor it to sound fresh, not copied.
Keep it concise (2-4 sentences). Do not include any thinking or reasoning."""),
                HumanMessage(content=f"RFP Question: {q}\n\nPast Answers for Reference:\n{past}")
            ])
            answer = msg.content
            if "<think>" in answer:
                answer = answer.split("</think>")[-1].strip()
            
            drafts.append({"question": q, "response": answer, "confidence": "high"})
            print(f"  ✅ Drafted: {q[:50]}...")
        else:
            # Flag for human review
            drafts.append({
                "question": q,
                "response": "[NEEDS HUMAN REVIEW — No sufficient prior answers found]",
                "confidence": "low"
            })
            flagged.append(q)
            print(f"  🚩 Flagged: {q[:50]}...")
    
    return {**state, "draft_responses": drafts, "flagged_items": flagged}

print("Node: draft_responses defined")

In [ ]:
# Node 4: Format final output
def format_output(state: RFPState) -> RFPState:
    print("\n📋 Formatting final output...")
    
    lines = ["=" * 70]
    lines.append("RFP RESPONSE DRAFT")
    lines.append("=" * 70)
    
    for i, draft in enumerate(state["draft_responses"]):
        confidence_badge = "✅" if draft["confidence"] == "high" else "🚩"
        lines.append(f"\nQ{i+1}: {draft['question']}")
        lines.append(f"[{confidence_badge} Confidence: {draft['confidence'].upper()}]")
        lines.append(f"\n{draft['response']}")
        lines.append("-" * 50)
    
    if state["flagged_items"]:
        lines.append(f"\n⚠️ {len(state['flagged_items'])} question(s) flagged for human review:")
        for f in state["flagged_items"]:
            lines.append(f"  • {f}")
    
    output = "\n".join(lines)
    return {**state, "final_output": output}

print("Node: format_output defined")

In [ ]:
# Build the LangGraph
graph = StateGraph(RFPState)

graph.add_node("parse_rfp", parse_rfp)
graph.add_node("retrieve_answers", retrieve_answers)
graph.add_node("draft_responses", draft_responses)
graph.add_node("format_output", format_output)

graph.set_entry_point("parse_rfp")
graph.add_edge("parse_rfp", "retrieve_answers")
graph.add_edge("retrieve_answers", "draft_responses")
graph.add_edge("draft_responses", "format_output")
graph.add_edge("format_output", END)

app = graph.compile()
print("LangGraph compiled!")
print("Nodes:", list(app.get_graph().nodes.keys()))

In [ ]:
# Run the RFP assistant
initial_state: RFPState = {
    "rfp_text": SAMPLE_RFP,
    "questions": [],
    "retrieved_context": [],
    "draft_responses": [],
    "flagged_items": [],
    "final_output": ""
}

result = app.invoke(initial_state)
print(result["final_output"])

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **RFP Parsing** | LLM extracts questions from RFP text |
| **RAG Retrieval** | ChromaDB finds similar past answers |
| **Confidence routing** | High-match → draft, low-match → flag |
| **LangGraph** | `StateGraph` with 4 sequential nodes |

## Architecture
```
RFP Text → parse_rfp → retrieve_answers → draft_responses → format_output → END
              ↓              ↓                   ↓
         Extract Qs    ChromaDB search     LLM draft or flag
```

## Extending This Project
- Add a human-in-the-loop node for flagged questions
- Include past RFP win/loss data to tailor tone
- Multi-section RFP support (pricing, legal, technical)
- Export to Word/PDF format